In [ ]:
#IMPORTS AND REQUIREMENTS

import os
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# 1. Hardware Configuration & Reproducibility
SEED = 42  #fixed seed for reproducibility
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

TRAIN_LABELS_CSV = "train_labels.csv"
SAMPLE_SUB_CSV = "sample_submission.csv"
TRAIN_IMG_DIR = os.path.join("dataset_images", "train")
TEST_IMG_DIR = os.path.join("dataset_images", "test")
CHECKPOINT_PATH = "best_resnet18.pt"

# 2. Labels & Stratified Splitting
train_df = pd.read_csv(TRAIN_LABELS_CSV, dtype={"image_id": str})
sample_sub = pd.read_csv(SAMPLE_SUB_CSV, dtype={"image_id": str})

CLASSES = sorted(train_df["class"].unique()) #get classes in alphabetical order
CLASS_TO_IDX = {cls_name: i for i, cls_name in enumerate(CLASSES)} #get class to index mapping, cateye -> 0 etc
IDX_TO_CLASS = {i: cls_name for cls_name, i in CLASS_TO_IDX.items()} #get index to class mapping, 0 -> cateye etc
train_df["target"] = train_df["class"].map(CLASS_TO_IDX) # map class names to indices for training

train_data, val_data = train_test_split(
    train_df, test_size=0.20, random_state=SEED, stratify=train_df["target"] # stratified split to maintain class distribution
)

# 3. Augmentation & Normalization Pipeline
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)), # resize images to 224x224 for ResNet-18
    transforms.RandomHorizontalFlip(p=0.5), # randomly flip images horizontally
    transforms.RandomRotation(degrees=10), # randomly rotate images
    transforms.ColorJitter(brightness=0.15, contrast=0.15), # randomly jitter image colors, these all help with generalization and prevent overfitting
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) #feature scaling to match ImageNet statistics, which ResNet-18 was trained on
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) #evaluation transforms should not include random augmentations, only resizing and normalization
])


# custom dataset class to handle image loading and transformations, including zero-padding for filenames
# 4. Dataset with zero-padding support (fixes leading zero filename issues)

class SafeDataset(Dataset):
    def __init__(self, df, img_dir, transform=None, is_test=False):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        raw_id = str(row["image_id"]).strip()
        base, ext = os.path.splitext(raw_id)
        ext = ext if ext else ".jpg"
        base_padded = base.zfill(5) if base.isdigit() else base

        filename = f"{base_padded}{ext}"
        path = os.path.join(self.img_dir, filename)
        if not os.path.exists(path):
            path = os.path.join(self.img_dir, f"{base}{ext}")

        image = Image.open(path).convert("RGB")
        if self.transform:
            image = self.transform(image)

        if self.is_test:
            return image, base_padded
        return image, torch.tensor(row["target"], dtype=torch.long)

train_loader = DataLoader(SafeDataset(train_data, TRAIN_IMG_DIR, train_transforms),
                          batch_size=32, shuffle=True, pin_memory=True, num_workers=2 if os.name != 'nt' else 0) # 
val_loader = DataLoader(SafeDataset(val_data, TRAIN_IMG_DIR, val_transforms),
                        batch_size=32, shuffle=False, pin_memory=True, num_workers=2 if os.name != 'nt' else 0)
test_loader = DataLoader(SafeDataset(sample_sub, TEST_IMG_DIR, val_transforms, is_test=True),
                         batch_size=32, shuffle=False, pin_memory=True, num_workers=2 if os.name != 'nt' else 0)

# 5. Load ResNet-18 and replace classification head
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT) # load pretrained ResNet-18 model with ImageNet weights
in_features = model.fc.in_features # get the number of input features to the final fully connected layer
model.fc = nn.Sequential( 
    nn.Dropout(p=0.3), # add dropout for regularization to prevent overfitting
    nn.Linear(in_features, len(CLASSES)) # replace the final layer with a new linear layer for our number of classes
) 
model = model.to(DEVICE) # move model to GPU if available

criterion = nn.CrossEntropyLoss(label_smoothing=0.05) # use label smoothing to prevent overconfidence in predictions and improve generalization
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-2) # use AdamW optimizer with weight decay
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=4) # use cosine annealing learning rate scheduler to gradually reduce the learning rate over epochs
scaler = torch.amp.GradScaler(enabled=torch.cuda.is_available()) # use automatic mixed precision for faster training and reduced memory usage on GPUs

# 6. Fine-Tuning Loop (4 Epochs)
best_val_acc = 0.0
for epoch in range(4):
    model.train()
    running_loss, corrects, total = 0.0, 0, 0
    for images, targets in tqdm(train_loader, desc=f"Epoch {epoch+1}/4 [Train]"):
        images, targets = images.to(DEVICE), targets.to(DEVICE)
        optimizer.zero_grad()

        with torch.amp.autocast(device_type="cuda" if torch.cuda.is_available() else "cpu"):
            outputs = model(images)
            loss = criterion(outputs, targets)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * targets.size(0)
        _, preds = outputs.max(1)
        corrects += (preds == targets).sum().item()
        total += targets.size(0)

    scheduler.step()
    train_acc = corrects / total

    # Validation
    model.eval() 
    val_corrects, val_total = 0, 0
    with torch.no_grad():
        for images, targets in val_loader:
            images, targets = images.to(DEVICE), targets.to(DEVICE)
            with torch.amp.autocast(device_type="cuda" if torch.cuda.is_available() else "cpu"):
                outputs = model(images)
            _, preds = outputs.max(1)
            val_corrects += (preds == targets).sum().item()
            val_total += targets.size(0)

    val_acc = val_corrects / val_total
    print(f"Epoch {epoch+1:02d} | Train Acc: {train_acc*100:.2f}% | Val Acc: {val_acc*100:.2f}%") # print training and validation accuracy for each epoch

    if val_acc > best_val_acc: # if current validation accuracy is better than the best so far, save the model checkpoint
        best_val_acc = val_acc
        torch.save(model.state_dict(), CHECKPOINT_PATH)
        print(f"  ⭐ Saved best model checkpoint (Val Acc: {best_val_acc*100:.2f}%)")

# 7. Generate Final Submission
print("\nGenerating Kaggle submission with best model...")
model.load_state_dict(torch.load(CHECKPOINT_PATH))
model.eval()

sub_ids, sub_preds = [], []
with torch.no_grad():
    for images, ids in tqdm(test_loader, desc="Inference"):
        images = images.to(DEVICE)
        with torch.amp.autocast(device_type="cuda" if torch.cuda.is_available() else "cpu"):
            outputs = model(images)
        _, preds = outputs.max(1)
        sub_ids.extend(ids)
        sub_preds.extend([IDX_TO_CLASS[p.item()] for p in preds])

submission_df = pd.DataFrame({"image_id": sub_ids, "class": sub_preds})
submission_df.to_csv("submission.csv", index=False)
print("Saved final predictions to submission.csv!")
print(submission_df.head())

c:\Users\ivorc\Downloads\ENGE707\kaggle\ENGE707-kaggle-comp\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\ivorc/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:04<00:00, 11.3MB/s]
Epoch 1/4 [Train]: 100%|██████████| 1592/1592 [14:36<00:00,  1.82it/s]


Epoch 01 | Train Acc: 96.44% | Val Acc: 98.31%
  ⭐ Saved best model checkpoint (Val Acc: 98.31%)


Epoch 2/4 [Train]: 100%|██████████| 1592/1592 [06:20<00:00,  4.18it/s]


...

Epoch 03 | Train Acc: 99.20% | Val Acc: 99.09%
  ⭐ Saved best model checkpoint (Val Acc: 99.09%)


Epoch 4/4 [Train]: 100%|██████████| 1592/1592 [06:38<00:00,  4.00it/s]


Epoch 04 | Train Acc: 99.65% | Val Acc: 99.51%
  ⭐ Saved best model checkpoint (Val Acc: 99.51%)

Generating Kaggle submission with best model...


Inference: 100%|██████████| 921/921 [06:36<00:00,  2.32it/s]

Saved final predictions to submission.csv!
  image_id     class
0    00001    DogEye
1    00002  DogMouth
2    00003   DogNose
3    00004   DogFace
4    00005  CatMouth


In [2]:
import torch.nn.functional as F

print("Generating predictions with Test-Time Augmentation (TTA)...")
model.load_state_dict(torch.load(CHECKPOINT_PATH))
model.eval()

sub_ids = []
sub_preds = []

with torch.no_grad():
    for images, ids in tqdm(test_loader, desc="TTA Inference"):
        images = images.to(DEVICE)
        
        # 1. Original image prediction
        with torch.amp.autocast(device_type="cuda" if torch.cuda.is_available() else "cpu"):
            outputs_orig = model(images)
            probs_orig = F.softmax(outputs_orig, dim=1)
            
            # 2. Horizontally flipped image prediction
            images_flipped = torch.flip(images, dims=[3])  # Flip width axis
            outputs_flip = model(images_flipped)
            probs_flip = F.softmax(outputs_flip, dim=1)
            
            # 3. Average the probabilities
            avg_probs = (probs_orig + probs_flip) / 2.0
            
        _, preds = torch.max(avg_probs, 1)
        
        sub_ids.extend(ids)
        sub_preds.extend([IDX_TO_CLASS[p.item()] for p in preds])

submission_df = pd.DataFrame({"image_id": sub_ids, "class": sub_preds})
submission_df.to_csv("submission_tta.csv", index=False)
print("Saved TTA predictions to submission_tta.csv! Submit this to Kaggle.")

Generating predictions with Test-Time Augmentation (TTA)...


TTA Inference: 100%|██████████| 921/921 [03:08<00:00,  4.88it/s]

Saved TTA predictions to submission_tta.csv! Submit this to Kaggle.
